# 第 32 课｜NumPy 广播与收益矩阵

本 Notebook 对应「365 天量化金融」第 32 课。课程代码主线来自本地视频、课件截图和 Whisper 口播整理；在此基础上补充量化数据语义、断言、性能验证和方法边界。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/32_youtube-309_C1gQHX8/index.html`
- 环境：`pkm-quant`
- 数据：固定随机种子生成的离线模拟行情（**非真实股票数据**）
- 状态：可运行学习稿；不代表策略收益承诺

> **数据声明**：本 Notebook 的证券名称、价格和成交量均为教学模拟数据。它用于稳定复现计算方法，不能用于评价真实证券、市场规律或策略收益。

## 本节量化问题

课程用价格数组和收益计算说明 ndarray、广播、矢量化、视图与拷贝。这里把示例扩展为“日期 × 股票”的价格与收益矩阵，重点覆盖日收益、等权横截面均值、超额收益、z-score、axis、性能和内存关系。

## 本节数据是什么？

这里读取的 `daily_prices.csv` 是**模拟行情，不是真实股票历史数据**：

- 6 只虚构股票：金融、新能源、公用事业、能源和科技行业；
- 756 个工作日，使用固定随机种子 `365`，每次生成结果一致；
- 收益由“市场共同波动 + 行业波动 + 个股随机波动”组合而成，因此保留了一些适合教学的相关性结构；
- 股票名称、OHLCV、价格路径和收益都不对应任何真实证券。

本节目标是验证 NumPy 的 shape、axis、广播和向量化是否正确。它不能用于评价真实市场、选择股票或证明策略有效；真实行情接口和数据快照将在数据获取课程中单独接入。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


## 先看原始行情表：一行代表什么？

`daily_prices.csv` 是“长表”（long format）：一行代表某个交易日、某只股票的一组行情。除了价格，还保留 `open/high/low/volume/sector` 等字段。
先读入并查看几行，确认我们还没有把数据变成矩阵。

In [2]:
# 读取原始长表，并把 date 解析为日期类型，先确认字段和数据形状。
daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
print("原始表形状：", daily.shape)
print("原始列：", daily.columns.tolist())
daily.head()

原始表形状： (4536, 9)
原始列： ['date', 'symbol', 'sector', 'open', 'high', 'low', 'close', 'adj_close', 'volume']


,date,symbol,sector,open,high,low,close,adj_close,volume
0,2023-01-03,BANK_A,金融,11.7051,11.7522,11.6747,11.7218,11.7218,2822797
1,2023-01-04,BANK_A,金融,11.7142,11.7878,11.6199,11.6934,11.6934,1999548
2,2023-01-05,BANK_A,金融,11.6517,11.7478,11.5905,11.6864,11.6864,1974821
3,2023-01-06,BANK_A,金融,11.6692,11.7522,11.5632,11.6461,11.6461,3999555
4,2023-01-09,BANK_A,金融,11.6141,11.6557,11.4507,11.4919,11.4919,2813450


## 用 `pivot` 把长表变成“日期 × 股票”的价格矩阵

这三行分别完成不同工作：

| 代码 | 含义 |
|---|---|
| `index="date"` | 把交易日放到行索引；一行对应一天 |
| `columns="symbol"` | 把股票代码展开成列；一列对应一只股票 |
| `values="adj_close"` | **只选择复权收盘价字段**，不是选择某一只股票 |
| `sort_index()` | 按日期从早到晚排序 |
| `to_numpy()` | 取出矩阵中的数值，丢掉日期和股票标签，交给 NumPy 做高速计算 |

因此，`close` 是带有日期和股票标签的 Pandas 表；`prices` 是包含**全部 6 只股票复权收盘价**的 NumPy 数组，不是单独一列价格。

`pivot` 不做求平均等聚合，要求每个“日期 × 股票”组合只有一行；如果原始数据有重复记录，应先清洗或明确使用 `pivot_table` 的聚合规则。

In [3]:
# 将长表重排为“日期 × 股票”的复权收盘价表，再转成 NumPy 数值矩阵。
close = daily.pivot(
    index="date",
    columns="symbol",
    values="adj_close",
).sort_index()

prices = close.to_numpy(dtype=np.float64)

print("close.shape =", close.shape, "含义：日期 × 股票")
print("prices.dtype =", prices.dtype)
assert close.index.is_unique and close.columns.is_unique
assert prices.shape == (close.shape[0], close.shape[1])
close.head()

close.shape = (756, 6) 含义：日期 × 股票
prices.dtype = float64


symbol,BANK_A,ENERGY_A,EV_A,INSURE_A,POWER_A,TECH_A
date,,,,,,
2023-01-03,11.7218,6.0296,27.5575,41.7118,8.0817,36.2134
2023-01-04,11.6934,6.0138,27.5189,41.9968,8.1296,36.5943
2023-01-05,11.6864,6.0281,27.8166,41.7611,8.1630,36.0263
2023-01-06,11.6461,6.1016,28.7476,42.6564,8.1751,35.9828
2023-01-09,11.4919,6.0260,28.3860,41.8431,8.1925,35.7220


## 课程示例 1：整排计算日收益

这里的公式是“今天价格 ÷ 昨天价格 − 1”。代码拆成四步书写，以便直接观察两个切片如何一一对应；运算顺序仍然是先做除法，再减去 `1`。

- `prices[1:]`：从第 2 个交易日开始，到最后一天；
- `prices[:-1]`：从第 1 天开始，到倒数第 2 天；
- 两个切片形状相同，NumPy 逐元素相除；
- `- 1` 把“价格比值”转换成“收益率”。例如 `105 / 100 - 1 = 0.05`，即 5%。

所以结果比原始价格少一行：第一个交易日没有前一天价格，无法计算日收益。

In [4]:
# 用相邻两天的价格计算日收益；拆开写便于观察切片、除法和减 1 的含义。
today_prices = prices[1:]
previous_prices = prices[:-1]
price_ratio = today_prices / previous_prices
returns_np = price_ratio - 1

assert returns_np.shape == (len(close) - 1, close.shape[1])
returns_table = pd.DataFrame(
    returns_np,
    index=close.index[1:],
    columns=close.columns,
)
returns_table.head()

symbol,BANK_A,ENERGY_A,EV_A,INSURE_A,POWER_A,TECH_A
date,,,,,,
2023-01-04,-0.002423,-0.002620,-0.001401,0.006833,0.005927,0.010518
2023-01-05,-0.000599,0.002378,0.010818,-0.005612,0.004108,-0.015522
2023-01-06,-0.003448,0.012193,0.033469,0.021439,0.001482,-0.001207
2023-01-09,-0.013240,-0.012390,-0.012578,-0.019066,0.002128,-0.007248
2023-01-10,-0.000948,0.004182,-0.004978,-0.031666,0.000110,0.004622


### 用 pandas 独立校验 NumPy 结果

`close.pct_change(fill_method=None)` 按每只股票列计算相邻两行的百分比变化。第一行没有前值，因此得到 `NaN`；`.iloc[1:]` 去掉这一行。`fill_method=None` 明确要求 pandas 不先用旧价格填补缺失值。

`np.testing.assert_allclose` 比较两个浮点数组是否在容许误差内一致；验证成功时没有输出，所以最后显式打印校验结果。

In [5]:
# 用带标签的 pandas 结果校验 NumPy 手工公式。
returns_pd = close.pct_change(fill_method=None).iloc[1:]
np.testing.assert_allclose(
    returns_np,
    returns_pd.to_numpy(),
    rtol=1e-12,
)
print("校验通过：NumPy 与 pandas 的日收益结果一致。")

校验通过：NumPy 与 pandas 的日收益结果一致。


## 课程示例 2：广播计算超额收益和横截面标准化

这里没有使用外部市场指数，而是把样本中 6 只股票每天的**等权横截面均值**当作教学代理。它每天只有一个值，个股收益每天有多个值。

`mean(axis=1)` 横向计算每一天的股票均值；`keepdims=True` 不需要额外 `reshape`，直接把结果保留为 `(T, 1)`，使它能沿股票维度广播。`std(..., ddof=0)` 在每天的 6 只样本股票上计算总体标准差，本例只用于演示横截面标准化。

In [6]:
# 复用同一个横截面均值，计算超额收益和每日横截面 z-score。
cross_mean = returns_np.mean(axis=1, keepdims=True)
cross_std = returns_np.std(axis=1, ddof=0, keepdims=True)
assert (cross_std > 0).all(), "存在横截面标准差为 0 的交易日，无法计算 z-score"

excess_return = returns_np - cross_mean
zscore = (returns_np - cross_mean) / cross_std

assert cross_mean.shape == (returns_np.shape[0], 1)
assert cross_std.shape == (returns_np.shape[0], 1)
np.testing.assert_allclose(excess_return.mean(axis=1), 0, atol=1e-15)
np.testing.assert_allclose(zscore.mean(axis=1), 0, atol=1e-12)
print(
    "shape:",
    "returns", returns_np.shape,
    "mean", cross_mean.shape,
    "std", cross_std.shape,
    "zscore", zscore.shape,
)

# 把第一个交易日的计算拆成表格，直接观察“个股收益−市场均值”和 z-score。
first_day_cross_section = pd.DataFrame(
    {
        "return": returns_np[0],
        "equal_weight_mean": cross_mean[0, 0],
        "excess_return": excess_return[0],
        "zscore": zscore[0],
    },
    index=close.columns,
)
first_day_cross_section.round(4)

shape: returns (755, 6) mean (755, 1) std (755, 1) zscore (755, 6)


,return,equal_weight_mean,excess_return,zscore
symbol,,,,
BANK_A,-0.0024,0.0028,-0.0052,-1.0128
ENERGY_A,-0.0026,0.0028,-0.0054,-1.0510
EV_A,-0.0014,0.0028,-0.0042,-0.8148
INSURE_A,0.0068,0.0028,0.0040,0.7800
POWER_A,0.0059,0.0028,0.0031,0.6046
TECH_A,0.0105,0.0028,0.0077,1.4939


## 轴选错不会报错，但金融含义会错

收益矩阵的行是交易日、列是股票：

- `axis=1`：压缩列，得到“每天跨股票”的统计量；
- `axis=0`：压缩行，得到“每只股票跨日期”的统计量。

NumPy 只执行指定方向，不知道你想算市场平均还是股票长期平均，因此必须先写清矩阵语义。

`axis` 可以理解为“被压缩掉的维度”。先用一个 `2 × 3` 小矩阵得到确定结果，再回到真实收益矩阵。

In [7]:
# 小矩阵：axis=1 每行一个均值；axis=0 每列一个均值。
axis_demo = np.array([[1, 2, 3], [4, 5, 6]])
print("axis=1:", axis_demo.mean(axis=1))  # [2., 5.]
print("axis=0:", axis_demo.mean(axis=0))  # [2.5, 3.5, 4.5]

# 回到收益矩阵，对比每天横截面均值和每只股票时间序列均值。
mean_by_date = returns_np.mean(axis=1)
mean_by_stock = returns_np.mean(axis=0)
assert mean_by_date.shape[0] == len(close) - 1
assert mean_by_stock.shape[0] == close.shape[1]

print("按日平均 shape:", mean_by_date.shape, "前 3 日:", np.round(mean_by_date[:3], 6))
pd.Series(mean_by_stock, index=close.columns, name="mean_daily_return").to_frame()

axis=1: [2. 5.]
axis=0: [2.5 3.5 4.5]
按日平均 shape: (755,) 前 3 日: [ 0.002806 -0.000738  0.010655]


,mean_daily_return
symbol,
BANK_A,0.000389
ENERGY_A,0.000124
EV_A,0.000614
INSURE_A,0.000501
POWER_A,0.000184
TECH_A,0.001154


## 课程示例 3：循环与矢量化性能

`np.tile(prices, (150, 1))` 只是把同一价格矩阵沿时间方向重复 150 次，用来放大性能差异；它没有产生新的市场信息，不能作为回测样本扩充方法。

两个函数都接收 `(T, N)` 价格矩阵，输出 `(T-1, N)` 收益矩阵：

- `np.empty` 只分配内存、不初始化内容；循环随后会覆盖每个位置；
- `timeit.repeat(..., number=1, repeat=3)` 独立计时 3 次，每次运行函数 1 次；
- 取最小值用于减少偶发系统调度干扰，只代表本机本次测量。

先用 `assert_allclose` 验证结果一致，再比较速度；否则“更快”没有意义。

In [8]:
# 用同一收益任务比较双重循环和 NumPy 切片矢量化的运行时间。
large = np.tile(prices, (150, 1))
print("性能测试矩阵:", prices.shape, "→", large.shape)

def loop_returns(array):
    # 输入 (T,N) 价格矩阵，输出 (T-1,N) 收益矩阵。
    output = np.empty((array.shape[0] - 1, array.shape[1]))
    for row in range(1, array.shape[0]):
        for column in range(array.shape[1]):
            output[row - 1, column] = array[row, column] / array[row - 1, column] - 1
    return output

def vectorized_returns(array):
    # 使用 NumPy 切片一次计算全部相邻收益。
    return array[1:] / array[:-1] - 1

np.testing.assert_allclose(loop_returns(large[:100]), vectorized_returns(large[:100]))
loop_time = min(timeit.repeat(lambda: loop_returns(large), number=1, repeat=3))
vector_time = min(timeit.repeat(lambda: vectorized_returns(large), number=1, repeat=3))
print(f"循环: {loop_time:.4f}s")
print(f"矢量化: {vector_time:.4f}s")
print(f"本机加速倍数: {loop_time / vector_time:.1f}x")

性能测试矩阵: (756, 6) → (113400, 6)


循环: 0.1148s
矢量化: 0.0005s
本机加速倍数: 245.0x


## 课程示例 4：视图、拷贝和常用数组变换

这组 API 分成三类学习：

1. `view` 与 `copy`：修改结果会不会影响原数组；
2. `reshape`、`concatenate`、`stack`：如何改变或组合矩阵形状；
3. `where` 与 `mean(axis=...)`：如何按条件筛选和按轴聚合。

### 4.1 切片视图与显式拷贝

NumPy 的**基础切片**通常返回视图，和原数组共享底层内存；花式索引和布尔索引通常返回拷贝。`.copy()` 可以显式创建独立数组。量化研究中如果对收益切片做原地去极值或标准化，误改原矩阵会污染后续实验。

In [9]:
# 基础切片共享内存，修改 view 会同步改变 original。
original = np.arange(1.0, 7.0)  # [1., 2., 3., 4., 5., 6.]
view = original[1:4]            # [2., 3., 4.]
view[0] = 99
assert np.shares_memory(original, view)
print("修改 view 后 original:", original)  # [1., 99., 3., 4., 5., 6.]

# 用一份干净数组演示 copy，修改副本不会影响来源。
copy_source = np.arange(1.0, 7.0)   # [1., 2., 3., 4., 5., 6.]
safe_copy = copy_source[1:4].copy() # [2., 3., 4.]
safe_copy[0] = -1
assert not np.shares_memory(copy_source, safe_copy)
assert copy_source[1] == 2
print("修改 copy 后来源数组:", copy_source)
print("独立的 safe_copy:", safe_copy)      # [-1., 3., 4.]

修改 view 后 original: [ 1. 99.  3.  4.  5.  6.]
修改 copy 后来源数组: [1. 2. 3. 4. 5. 6.]
独立的 safe_copy: [-1.  3.  4.]


### 4.2 改变形状与组合数组

| 方法 | 做什么 | 本例结果 | 量化场景 |
|---|---|---|---|
| `reshape(2, 3)` | 不改元素，只改变观察形状 | `(6,) → (2, 3)` | 将一维数据按批次或资产维度组织 |
| `concatenate` | 沿已有轴首尾拼接 | `[1,2] + [3,4] → [1,2,3,4]` | 拼接连续时间段或同结构数据 |
| `stack` | 新建一个轴后堆叠 | 两个 `(3,) → (3,2)` | 组成“日期 × 股票”矩阵 |

`reshape` 前后元素总数必须一致；`concatenate` 默认沿 `axis=0` 拼接且不新增维度，其他轴的长度必须兼容；`stack` 会先新增维度。

In [10]:
# 分别演示改变形状、沿已有轴拼接、沿新轴堆叠。
source = np.arange(6)                     # [0, 1, 2, 3, 4, 5]
reshaped = source.reshape(2, 3)           # [[0,1,2], [3,4,5]]

left = np.array([1, 2])
right = np.array([3, 4])
concatenated = np.concatenate([left, right])  # [1, 2, 3, 4]

stock_a_returns = returns_np[:3, 0]       # 股票 A 的前 3 日收益
stock_b_returns = returns_np[:3, 1]       # 股票 B 的前 3 日收益
stacked = np.stack(
    [stock_a_returns, stock_b_returns],
    axis=1,
)  # shape=(3,2)，行=日期，列=股票

print("source:", source, "shape =", source.shape)
print("reshape:\n", reshaped, "shape =", reshaped.shape)
print("concatenate:", concatenated)
print("stack:\n", np.round(stacked, 6), "shape =", stacked.shape)

source: [0 1 2 3 4 5] shape = (6,)
reshape:
 [[0 1 2]
 [3 4 5]] shape = (2, 3)
concatenate: [1 2 3 4]
stack:
 [[-0.002423 -0.00262 ]
 [-0.000599  0.002378]
 [-0.003448  0.012193]] shape = (3, 2)


### 4.3 条件选择与按股票求均值

`np.where(condition, x, y)` 对每个位置判断条件：满足时取 `x`，否则取 `y`，输出 shape 与广播后的输入一致。它不会删除元素；真正筛除元素要使用布尔索引。这里把首日负收益替换成 0，只用于演示条件选择，不代表可交易策略。

`returns_np.mean(axis=0)` 压缩时间轴，得到每只股票一个样本期平均日收益。它是描述统计，不等于未来预期收益。

In [11]:
# 首日只保留正收益；再沿时间轴计算每只股票的平均日收益。
first_day_returns = returns_np[0]
positive_only = np.where(first_day_returns > 0, first_day_returns, 0)
mean_return = returns_np.mean(axis=0)

pd.DataFrame(
    {
        "first_day_return": first_day_returns,
        "positive_only": positive_only,
        "sample_mean_return": mean_return,
    },
    index=close.columns,
).round(6)

,first_day_return,positive_only,sample_mean_return
symbol,,,
BANK_A,-0.002423,0.000000,0.000389
ENERGY_A,-0.002620,0.000000,0.000124
EV_A,-0.001401,0.000000,0.000614
INSURE_A,0.006833,0.006833,0.000501
POWER_A,0.005927,0.005927,0.000184
TECH_A,0.010518,0.010518,0.001154


## 方法边界

- 广播不理解证券标签，矩阵进入 NumPy 前要由 Pandas 完成对齐。
- 向量化可能创建大型中间数组；内存峰值也要测量。
- 金额、累计净值和优化问题默认保留 `float64`，不要只为省内存盲目降精度。
- 性能倍数与硬件、数据规模有关，以本机实测为准，不照搬课程中的固定倍数。